In [171]:
from binance_websocket import BinanceWebsocket
import pandas as pd
from upbit_websocket import UpbitWebsocket
import time
import datetime
from multiprocessing import Process, Manager
import pickle

# Go to the upper directory with absolute path
import sys
sys.path.append('/home/kp_info_loader')
from etc.redis_connector.redis_connector import InitRedis

In [2]:
binance_spot_symbol_list = ['BTCUSDT','ETHUSDT','XRPUSDT','BCHUSDT','XEMUSDT','MTLUSDT','KAVAUSDT','TUSDT','SUIUSDT','SOLUSDT','WAVESUSDT','STXUSDT','STORJUSDT','DOGEUSDT','ETCUSDT','FLOWUSDT','ETCUSDT','MATICUSDT','AAVEUSDT','ALGOUSDT','GMTUSDT','SANDUSDT','ANKRUSDT','ONTUSDT','SXPUSDT']
binance_usdm_symbol_list = binance_spot_symbol_list
binance_coinm_symbol_list = [x[:-1]+'_PERP' for x in binance_spot_symbol_list]
proc_n = 3
logging_dir = '/home/kp_info_loader/exchange_websocket/'
binance_ws = BinanceWebsocket(binance_spot_symbol_list, binance_usdm_symbol_list, binance_coinm_symbol_list, proc_n, logging_dir)

2023-07-17 15:05:26,174 - binance_websocket - INFO - started 1th Binance price websocket for usdm, coinm, spot proc..
2023-07-17 15:05:26,283 - binance_websocket - INFO - started 2th Binance price websocket for usdm, coinm, spot proc..
2023-07-17 15:05:26,394 - binance_websocket - INFO - started 3th Binance price websocket for usdm, coinm, spot proc..


In [3]:
upbit_symbol_list = ['KRW-'+x[:-4] for x in binance_spot_symbol_list]
upbit_ws = UpbitWebsocket(upbit_symbol_list, proc_n, logging_dir)

2023-07-15 14:20:03,848 - upbit_websocket - INFO - upbit_orderbook_ticker_websocket|1th upbit_ticker_proc started.


2023-07-15 14:20:03,952 - upbit_websocket - INFO - upbit_websocket|upbit_websocket started
2023-07-15 14:20:04,360 - upbit_websocket - INFO - upbit_orderbook_ticker_websocket|1th upbit_orderbook_proc started.
2023-07-15 14:20:04,449 - upbit_websocket - INFO - upbit_websocket|upbit_websocket started
2023-07-15 14:20:04,870 - upbit_websocket - INFO - upbit_orderbook_ticker_websocket|2th upbit_ticker_proc started.
2023-07-15 14:20:04,959 - upbit_websocket - INFO - upbit_websocket|upbit_websocket started
2023-07-15 14:20:05,378 - upbit_websocket - INFO - upbit_orderbook_ticker_websocket|2th upbit_orderbook_proc started.
2023-07-15 14:20:05,463 - upbit_websocket - INFO - upbit_websocket|upbit_websocket started
2023-07-15 14:20:05,890 - upbit_websocket - INFO - upbit_orderbook_ticker_websocket|3th upbit_ticker_proc started.
2023-07-15 14:20:05,987 - upbit_websocket - INFO - upbit_websocket|upbit_websocket started
2023-07-15 14:20:06,401 - upbit_websocket - INFO - upbit_orderbook_ticker_webso

2023-07-17 15:05:26,519 - binance_websocket - INFO - started binance_liquidation_websocket for usdm, coinm..
2023-07-17 15:05:26,522 - binance_websocket - INFO - started binance_kline_websocket for usdm, coinm, spot..


2023-07-17 15:05:26,548 - upbit_websocket - INFO - upbit_orderbook_ticker_websocket|1th upbit_ticker_proc started.
2023-07-17 15:05:26,657 - upbit_websocket - INFO - upbit_websocket|upbit_websocket started
2023-07-17 15:05:27,061 - upbit_websocket - INFO - upbit_orderbook_ticker_websocket|1th upbit_orderbook_proc started.
2023-07-17 15:05:27,156 - upbit_websocket - INFO - upbit_websocket|upbit_websocket started
2023-07-17 15:05:27,574 - upbit_websocket - INFO - upbit_orderbook_ticker_websocket|2th upbit_ticker_proc started.
2023-07-17 15:05:27,657 - upbit_websocket - INFO - upbit_websocket|upbit_websocket started
2023-07-17 15:05:28,085 - upbit_websocket - INFO - upbit_orderbook_ticker_websocket|2th upbit_orderbook_proc started.
2023-07-17 15:05:28,169 - upbit_websocket - INFO - upbit_websocket|upbit_websocket started
2023-07-17 15:05:28,596 - upbit_websocket - INFO - upbit_orderbook_ticker_websocket|3th upbit_ticker_proc started.
2023-07-17 15:05:28,688 - upbit_websocket - INFO - upbi

In [4]:
def get_ticker_ratio(price_krw):
    if price_krw < 0.1:
        ticker_size = 0.0001
    elif price_krw < 1:
        ticker_size = 0.001
    elif price_krw < 10:
        ticker_size = 0.01
    elif price_krw < 100:
        ticker_size = 0.1
    elif price_krw < 1000:
        ticker_size = 1
    elif price_krw < 10000:
        ticker_size = 5
    elif price_krw < 100000:
        ticker_size = 10
    elif price_krw < 500000:
        ticker_size = 50
    elif price_krw < 1000000:
        ticker_size = 100
    elif price_krw < 2000000:
        ticker_size = 500
    elif price_krw >= 2000000:
        ticker_size = 1000
    ticker_ratio = ticker_size / price_krw
    return ticker_ratio

def binance_bookticker_convert(BINANCE_BOOKTICKER_DICT):
    BINANCE_BOOKTICKER_DICT_copy = dict(BINANCE_BOOKTICKER_DICT.copy())
    converted_df = pd.DataFrame(BINANCE_BOOKTICKER_DICT_copy).transpose().reset_index(drop=True)
    # binance_bookTicker_column_dict = {
    #     'e':'event_type',
    #     'u':'order_book_updateId',
    #     'E':'event_time',
    #     'T':'transaction_time',
    #     's':'symbol',
    #     'b':'best_bid_price',
    #     'B':'best_bid_qty',
    #     'a':'best_ask_price',
    #     'A':'best_ask_qty'
    # }
    # temp_df = temp_df.rename(columns=binance_bookTicker_column_dict)
    return converted_df

def binance_ticker_convert(binance_ticker_dict):
    binance_ticker_dict_copy = dict(binance_ticker_dict.copy())
    converted_df = pd.DataFrame(binance_ticker_dict_copy).transpose().reset_index(drop=True)
    return converted_df

def upbit_ticker_convert(UPBIT_TICKER_DICT):
    UPBIT_TICKER_DICT_copy = dict(UPBIT_TICKER_DICT.copy())
    converted_df = pd.DataFrame(UPBIT_TICKER_DICT_copy).transpose().reset_index(drop=True)
    # upbit_column_dict = {
    #     'ty':'type',
    #     'cd':'code',
    #     'op':'opening_price',
    #     'hp':'high_price',
    #     'lp':'low_price',
    #     'tp':'trade_price',
    #     'pcp':'prev_closing_price',
    #     'c':'change',
    #     'cp':'change_price',
    #     'scp':'signed_change_price',
    #     'cr':'change_rate',
    #     'scr':'signed_change_rate',
    #     'tv':'trade_volume',
    #     'atv':'acc_trade_volume',
    #     'atv24h':'acc_trade_volume_24h',
    #     'atp':'acc_trade_price',
    #     'atp24h':'acc_trade_price_24h',
    #     'tdt':'trade_date',
    #     'ttm':'trade_time',
    #     'ttms':'trade_timestamp',
    #     'ab':'ask_bid',
    #     'aav':'acc_ask_volume',
    #     'abv':'acc_bid_volume',
    #     'h52wp':'highest_52_week_price',
    #     'h52wdt':'highest_52_week_date',
    #     'l52wp':'lowest_52_week_price',
    #     'l52wdt':'lowest_52_week_date',
    #     'ts':'trade_status',
    #     'ms':'market_state',
    #     'msfi':'market_state_for_ios',
    #     'its':'is_trading_suspended',
    #     'dd':'delisting_date',
    #     'mw':'market_warning',
    #     'tms':'timestamp',
    #     'st':'stream_type'
    # }
    # temp_df = temp_df.rename(columns=upbit_column_dict)
    return converted_df

def upbit_orderbook_convert(UPBIT_ORDERBOOK_DICT):
    UPBIT_ORDERBOOK_DICT_copy = dict(UPBIT_ORDERBOOK_DICT.copy())
    converted_df = pd.DataFrame(UPBIT_ORDERBOOK_DICT_copy).transpose().reset_index(drop=True)
    # upbit_column_dict = {
    #     'ty':'type',
    #     'cd':'code',
    #     'tas':'total_ask_size',
    #     'tbs':'total_bid_size',
    #     'obu':'orderbook_units',
    #     'tms':'timestamp',
    #     'st':'stream_type'
    # }
    # temp_df = temp_df.rename(columns=upbit_column_dict)
    return converted_df

In [5]:
binance_bookticker_convert(binance_ws.binance_usdm_bookticker_dict)

,e,u,s,b,B,a,A,T,E
0,bookTicker,3070568036178,BTCUSDT,30252.80,22.844,30252.90,11.245,1689573929605,1689573929618
1,bookTicker,3070568036400,BCHUSDT,251.31,13.136,251.32,23.106,1689573929614,1689573929619
2,bookTicker,3070568036407,DOGEUSDT,0.071490,69535,0.071500,186276,1689573929614,1689573929619
3,bookTicker,3070568036562,ETHUSDT,1929.68,212.006,1929.69,10.040,1689573929618,1689573929622
4,bookTicker,3070568028638,WAVESUSDT,2.0072,664.1,2.0073,139.3,1689573929352,1689573929356
5,bookTicker,3070568014978,STORJUSDT,0.3142,46587,0.3143,7155,1689573928959,1689573928962
6,bookTicker,3070568031368,SANDUSDT,0.45120,21348,0.45130,14498,1689573929454,1689573929459
7,bookTicker,3070567999054,TUSDT,0.0241200,21776,0.0241300,212732,1689573928402,1689573928407
8,bookTicker,3070568031272,MATICUSDT,0.77800,7729,0.77810,25434,1689573929451,1689573929456
9,bookTicker,3070568031805,FLOWUSDT,0.623,94196.5,0.624,98929.8,1689573929470,1689573929474


In [6]:
temp = binance_ticker_convert(binance_ws.binance_usdm_ticker_dict)
temp[temp['s']=='BTCUSDT'][['v','q']]

,v,q
6,149146.257,4517584656.84


In [7]:
upbit_orderbook_convert(upbit_ws.upbit_orderbook_dict)

,ty,cd,tms,tas,tbs,obu,st
0,orderbook,KRW-BTC,1689573930496,6.344443,8.738829,"[{'ap': 38930000, 'bp': 38909000, 'as': 0.0492...",REALTIME
1,orderbook,KRW-BCH,1689573931242,654.98349,708.358173,"[{'ap': 323250, 'bp': 323200, 'as': 11.4322128...",REALTIME
2,orderbook,KRW-KAVA,1689573906445,998393.111561,944659.215532,"[{'ap': 1190, 'bp': 1185, 'as': 13613.42502966...",SNAPSHOT
3,orderbook,KRW-SOL,1689573931199,10281.672172,12482.718073,"[{'ap': 35780, 'bp': 35750, 'as': 405.95736918...",REALTIME
4,orderbook,KRW-STORJ,1689573931019,1599663.079853,1284935.846228,"[{'ap': 404, 'bp': 403, 'as': 50824.49010232, ...",REALTIME
5,orderbook,KRW-FLOW,1689573929621,518348.894494,411313.195671,"[{'ap': 803, 'bp': 802, 'as': 6697.95310942, '...",REALTIME
6,orderbook,KRW-AAVE,1689573931252,1384.86779,1159.986969,"[{'ap': 101450, 'bp': 101250, 'as': 7.02189256...",REALTIME
7,orderbook,KRW-SAND,1689573929983,760199.378035,1050307.582169,"[{'ap': 580, 'bp': 579, 'as': 28601.94189938, ...",REALTIME
8,orderbook,KRW-SXP,1689573931241,1135159.776816,1054268.27214,"[{'ap': 481, 'bp': 480, 'as': 39172.26382353, ...",REALTIME
9,orderbook,KRW-ETH,1689573924943,303.329202,260.416198,"[{'ap': 2483000, 'bp': 2482000, 'as': 19.60044...",SNAPSHOT


In [8]:
upbit_ticker_convert(upbit_ws.upbit_ticker_dict)

,ty,cd,op,hp,lp,tp,pcp,atp,c,cp,...,l52wp,l52wdt,ms,its,dd,mw,tms,atp24h,atv24h,st
0,ticker,KRW-BTC,38946000.0,38983000.0,38832000.0,38930000.0,38933000.0,21523413830.796768,FALL,3000.0,...,20700000.0,2022-12-30,ACTIVE,False,None,NONE,1689573933148,75850451820.101379,1947.765096,REALTIME
1,ticker,KRW-BCH,322150.0,327300.0,319050.0,323250.0,322100.0,12748012801.96859,RISE,1150.0,...,120450.0,2022-12-30,ACTIVE,False,None,NONE,1689573930169,38744161488.730171,118895.612067,REALTIME
2,ticker,KRW-KAVA,1190.0,1195.0,1175.0,1185.0,1190.0,2658997240.760039,FALL,5.0,...,960.0,2023-06-15,ACTIVE,False,None,NONE,1689573900330,6689661225.485402,5627455.378676,SNAPSHOT
3,ticker,KRW-SOL,35340.0,36620.0,35100.0,35750.0,35350.0,22099247710.811935,RISE,400.0,...,10140.0,2022-12-29,ACTIVE,False,None,NONE,1689573931039,65587740163.807823,1823183.842709,REALTIME
4,ticker,KRW-STORJ,398.0,404.0,394.0,404.0,398.0,2422131586.689389,RISE,6.0,...,282.0,2023-06-15,ACTIVE,False,None,NONE,1689573930899,10075053602.285778,25022267.886,REALTIME
5,ticker,KRW-FLOW,786.0,814.0,769.0,803.0,787.0,15920153142.46673,RISE,16.0,...,577.0,2023-06-15,ACTIVE,False,None,NONE,1689573930439,28558836545.623806,35909566.518973,REALTIME
6,ticker,KRW-AAVE,99160.0,102150.0,98370.0,101050.0,98900.0,1093522755.326471,RISE,2150.0,...,63160.0,2023-06-10,ACTIVE,False,None,NONE,1689573900497,2608349081.498388,25941.725465,SNAPSHOT
7,ticker,KRW-SAND,569.0,582.0,566.0,580.0,569.0,5034280269.945773,RISE,11.0,...,473.0,2023-06-15,ACTIVE,False,None,NONE,1689573931779,12340437956.202471,21422522.308907,REALTIME
8,ticker,KRW-SXP,477.0,483.0,474.0,481.0,477.0,2468999106.082999,RISE,4.0,...,248.0,2022-12-31,ACTIVE,False,None,NONE,1689573900372,6073160008.288413,12709764.704621,SNAPSHOT
9,ticker,KRW-ETH,2477000.0,2486000.0,2467000.0,2483000.0,2476000.0,6373265351.29406,RISE,7000.0,...,1500000.0,2022-12-30,ACTIVE,False,None,NONE,1689573930033,17270464634.997429,6961.78567,REALTIME


In [255]:
current_dollar = 1270

def generate_upbit_binance_kimp_df(upbit_ticker_dict, upbit_orderbook_dict, binance_ticker_dict, binance_bookticker_dict):
    upbit_allticker_df = upbit_ticker_convert(upbit_ticker_dict)
    upbit_orderbook_df = upbit_orderbook_convert(upbit_orderbook_dict)
    binance_ticker_df = binance_ticker_convert(binance_ticker_dict)
    binance_bookticker_df = binance_bookticker_convert(binance_bookticker_dict)
    upbit_allticker_df = upbit_allticker_df.merge(upbit_orderbook_df, on='cd')
    upbit_allticker_df['base_symbol'] = upbit_allticker_df['cd'].apply(lambda x: x.replace('KRW-', ''))

    binance_allticker_df = binance_bookticker_df.merge(binance_ticker_df[['s','c','q']], on='s')
    binance_allticker_df['base_symbol'] = binance_bookticker_df['s'].apply(lambda x: x.replace('USDT', '').replace('USD_PERP',''))
    merged_allticker_df = upbit_allticker_df.merge(binance_allticker_df, on='base_symbol')
    merged_allticker_df.loc[:, ['a','b','c_y','q']] = merged_allticker_df[['a','b','c_y','q']].astype(float) # 'c_y' is a trade_price of binance
    merged_allticker_df[['binance_bid_price_krw', 'binance_trade_price_krw', 'binance_ask_price_krw']] = merged_allticker_df[['b','c_y','a']] * current_dollar
    merged_allticker_df['upbit_ask_price'] = merged_allticker_df['obu'].apply(lambda x: x[0]['ap'])
    merged_allticker_df['upbit_bid_price'] = merged_allticker_df['obu'].apply(lambda x: x[0]['bp'])
    merged_allticker_df['enter_kimp'] = (merged_allticker_df['upbit_ask_price'] - merged_allticker_df['binance_bid_price_krw']) / merged_allticker_df['binance_bid_price_krw']
    merged_allticker_df['exit_kimp'] = (merged_allticker_df['upbit_bid_price'] - merged_allticker_df['binance_ask_price_krw']) / merged_allticker_df['binance_ask_price_krw']
    merged_allticker_df['tp_kimp'] = (merged_allticker_df['tp'] - merged_allticker_df['binance_trade_price_krw']) / merged_allticker_df['binance_trade_price_krw']
    merged_allticker_df['enter_usdt'] = (merged_allticker_df['enter_kimp'] + 1) * current_dollar
    merged_allticker_df['exit_usdt'] = (merged_allticker_df['exit_kimp'] + 1) * current_dollar
    merged_allticker_df['tp_usdt'] = (merged_allticker_df['tp_kimp'] + 1) * current_dollar
    merged_allticker_df['upbit_ticker_ratio'] = merged_allticker_df['tp'].apply(lambda x: get_ticker_ratio(x))
    merged_allticker_df = merged_allticker_df.rename(columns={
        's':'symbol',
        'atp24h':'upbit_acc_trade_price_24h',
        'q':'binance_acc_trade_price_24h_usdt',
        'tp':'upbit_trade_price',
        'b':'binance_bid_price',
        'a':'binance_ask_price',
        'c_y':'binance_trade_price',
        'scr':'signed_change_rate',
        'ms':'upbit_market_state',
        'dd':'upbit_delisting_date',
        'mw':'upbit_market_warning',
        'its':'upbit_is_trading_suspended',
        'tms_x':'upbit_timestamp',
        'E':'binance_bookticker_event_time'
    })
    # SPOT bookticker doesn't have event_time, but it's not a problem.
    if 'binance_bookticker_event_time' not in merged_allticker_df.columns:
        merged_allticker_df['binance_bookticker_event_time'] = -1
    merged_allticker_df['binance_acc_trade_price_24h_krw'] = merged_allticker_df['binance_acc_trade_price_24h_usdt'] * current_dollar
    kimp_df = (merged_allticker_df[['symbol','upbit_acc_trade_price_24h','upbit_trade_price','upbit_ticker_ratio','upbit_ask_price','upbit_bid_price','binance_acc_trade_price_24h_usdt', 'binance_acc_trade_price_24h_krw',
    'binance_bid_price','binance_bid_price_krw','binance_trade_price','binance_trade_price_krw','binance_ask_price','binance_ask_price_krw','signed_change_rate','upbit_market_state','upbit_delisting_date',
    'upbit_market_warning','upbit_is_trading_suspended','upbit_timestamp','binance_bookticker_event_time',
    'enter_kimp', 'exit_kimp', 'tp_kimp',
    'enter_usdt', 'exit_usdt', 'tp_usdt']])
    return kimp_df

def generate_ohlc_df(df, freq='1T'):
    df = df.set_index(['symbol', 'datetime_now'])
    if 'tp_kimp' in df.columns:
        df['tp_kimp'] = pd.to_numeric(df['tp_kimp'], errors='coerce')
        ohlc_df = df.groupby(['symbol', pd.Grouper(level='datetime_now', freq=freq)])['tp_kimp'].ohlc()
    elif 'enter_kimp' in df.columns:
        df['enter_kimp'] = pd.to_numeric(df['enter_kimp'], errors='coerce')
        ohlc_df = df.groupby(['symbol', pd.Grouper(level='datetime_now', freq=freq)])['enter_kimp'].ohlc()
    elif 'exit_kimp' in df.columns:
        df['exit_kimp'] = pd.to_numeric(df['exit_kimp'], errors='coerce')
        ohlc_df = df.groupby(['symbol', pd.Grouper(level='datetime_now', freq=freq)])['exit_kimp'].ohlc()
    else:
        raise ValueError('There is no kimp column in the dataframe')
    return ohlc_df


In [150]:
redis_cli = InitRedis()

In [196]:
fetched_df.columns

Index(['symbol', 'upbit_acc_trade_price_24h', 'upbit_trade_price',
       'upbit_ticker_ratio', 'upbit_ask_price', 'upbit_bid_price',
       'binance_acc_trade_price_24h_usdt', 'binance_acc_trade_price_24h_krw',
       'binance_bid_price', 'binance_bid_price_krw', 'binance_trade_price',
       'binance_trade_price_krw', 'binance_ask_price', 'binance_ask_price_krw',
       'signed_change_rate', 'upbit_market_state', 'upbit_delisting_date',
       'upbit_market_warning', 'upbit_is_trading_suspended', 'upbit_timestamp',
       'binance_bookticker_event_time', 'enter_kimp', 'exit_kimp', 'tp_kimp',
       'enter_usdt', 'exit_usdt', 'tp_usdt'],
      dtype='object')

In [199]:
60 / 0.02

3000.0

In [216]:
datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')

'2023-07-18 20:02:03.982928'

In [240]:
test_appended_kimp_df[test_appended_kimp_df['datetime_now']<'2023-08-31 00:00:00.000000']

,symbol,tp_kimp,tp_usdt,datetime_now
0,BTCUSDT,0.009003,1281.434055,2023-07-18 20:07:29.453044
1,BCHUSDT,0.008457,1280.739844,2023-07-18 20:07:29.453044
2,KAVAUSDT,0.00938,1281.912004,2023-07-18 20:07:29.453044
3,SOLUSDT,0.00781,1279.918327,2023-07-18 20:07:29.453044
4,STORJUSDT,0.007732,1279.819471,2023-07-18 20:07:29.453044
...,...,...,...,...
19,SUIUSDT,0.006627,1278.416689,2023-07-18 20:07:29.453044
20,STXUSDT,0.006757,1278.581428,2023-07-18 20:07:29.453044
21,MATICUSDT,0.008039,1280.209014,2023-07-18 20:07:29.453044
22,GMTUSDT,0.011624,1284.763024,2023-07-18 20:07:29.453044


In [266]:
 
# start = time.time()
# appended_kimp_df = pd.DataFrame()
# start_datetime = datetime.datetime.now()
# while True:
#     if datetime.datetime.now() - start_datetime > datetime.timedelta(minutes=1):
#         break
#     kimp_df = generate_upbit_binance_kimp_df(upbit_ws.upbit_ticker_dict, upbit_ws.upbit_orderbook_dict, binance_ws.binance_usdm_ticker_dict, binance_ws.binance_usdm_bookticker_dict)
#     kimp_df['datetime_now'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")
#     appended_kimp_df = pd.concat([appended_kimp_df, kimp_df], axis=0)

cut_i = 20000
column_list = ['symbol','tp_kimp','tp_usdt']
# print(f"generating_kimp_df: {time.time()-start}")
start = time.time()
test_appended_kimp_df = appended_kimp_df.iloc[:cut_i, :][column_list]
test_appended_kimp_df['datetime_now'] = datetime.datetime.now()#.strftime("%Y-%m-%d %H:%M:%S.%f")
# print(f"length of test_appended_kimp_df: {len(test_appended_kimp_df)}, size: {sys.getsizeof(test_appended_kimp_df) / 1024 / 1024}")
# appended_kimp_dict = test_appended_kimp_df.reset_index(drop=True).to_dict()
# print(f"converting df to dict: {time.time()-start}, size: {sys.getsizeof(appended_kimp_dict) / 1024 / 1024}")
# start = time.time()
# appended_kimp_pickle = pickle.dumps(appended_kimp_dict)
# print(f"converting df to pickle: {time.time()-start}, size: {sys.getsizeof(appended_kimp_pickle) / 1024 / 1024}")
# start = time.time()
# redis_cli.set_dict('kimp_dict', appended_kimp_dict)
# print(f"redis saving dict: {time.time()-start}")
# start = time.time()
# redis_cli.set_data('kimp_dict_pickle', appended_kimp_pickle)
# print(f"redis saving pickle: {time.time()-start}")
# start = time.time()
# fetched_dict = redis_cli.get_dict('kimp_dict')
# fetched_df = pd.DataFrame(fetched_dict)
# # fetched_df.loc[:, 'datetime_now'] = pd.to_datetime(fetched_df['datetime_now'])
# print(f"redis fetching dict: {time.time()-start}")
# start = time.time()
# fetched_pickle = redis_cli.get_data('kimp_dict_pickle')
# fetched_df = pd.DataFrame(pickle.loads(fetched_pickle))
# # fetched_df.loc[:, 'datetime_now'] = pd.to_datetime(fetched_df['datetime_now'])
# print(f"redis fetching pickle: {time.time()-start}")


In [267]:
len(test_appended_kimp_df)

20000

In [270]:
kimp_df = generate_upbit_binance_kimp_df(upbit_ws.upbit_ticker_dict, upbit_ws.upbit_orderbook_dict, binance_ws.binance_usdm_ticker_dict, binance_ws.binance_usdm_bookticker_dict)
kimp_df['datetime_now'] = datetime.datetime.now()#.strftime("%Y-%m-%d %H:%M:%S.%f")
start = time.time()
ohlc_df = generate_ohlc_df(test_appended_kimp_df, freq='1T')
print(f"generating ohlc df: {time.time()-start}")
ohlc_df

generating ohlc df: 0.015347003936767578


,,open,high,low,close
symbol,datetime_now,,,,
AAVEUSDT,2023-07-18 22:07:00,0.008971,0.009380,0.008835,0.009244
ALGOUSDT,2023-07-18 22:07:00,0.014280,0.015140,0.014280,0.015140
ANKRUSDT,2023-07-18 22:07:00,0.009961,0.009961,0.009961,0.009961
BCHUSDT,2023-07-18 22:07:00,0.008457,0.008878,0.008457,0.008878
BTCUSDT,2023-07-18 22:07:00,0.009003,0.009141,0.008947,0.009138
DOGEUSDT,2023-07-18 22:07:00,0.007865,0.008010,0.007720,0.008010
ETCUSDT,2023-07-18 22:07:00,0.008681,0.008893,0.008681,0.008840
ETHUSDT,2023-07-18 22:07:00,0.008445,0.008625,0.008434,0.008620
FLOWUSDT,2023-07-18 22:07:00,0.009770,0.009770,0.008554,0.008554


In [287]:
60/23000

0.0026086956521739132

In [299]:

def ohlc_loader(redis_cli, loop_downtime_sec=0.01):
    appended_kimp_df = pd.DataFrame()
    kline_switch = False
    while True:
        time.sleep(loop_downtime_sec)
        kimp_df = generate_upbit_binance_kimp_df(upbit_ws.upbit_ticker_dict, upbit_ws.upbit_orderbook_dict, binance_ws.binance_usdm_ticker_dict, binance_ws.binance_usdm_bookticker_dict)
        datetime_now = datetime.datetime.now()
        kimp_df['datetime_now'] = datetime_now
        appended_kimp_df = pd.concat([appended_kimp_df, kimp_df], axis=0)
        # print(f"redis saving ohlc_1m_now time: {time.time()-start}")
        if datetime_now.second == 0 and kline_switch == True:
            adjusted_datetime_now = datetime.datetime(datetime_now.year, datetime_now.month, datetime_now.day, datetime_now.hour, datetime_now.minute)
            cut_appended_kimp_df = appended_kimp_df[appended_kimp_df['datetime_now'] < adjusted_datetime_now]
            start = time.time()
            ohlc_df = generate_ohlc_df(cut_appended_kimp_df)
            print(f"generating ohlc_df(length: {len(cut_appended_kimp_df)}): {time.time()-start}")
            # INSERT into redis db for current data
            start = time.time()
            redis_cli.set_data('ohlc_1m_now', pickle.dumps(ohlc_df))
            print(f"redis saving ohlc_1m_now time: {time.time()-start}")
            # Append into redis db for historical data
            start = time.time()
            old_ohlc_1m_kline = redis_cli.get_data('ohlc_1m_kline')
            if old_ohlc_1m_kline is None:
                old_ohlc_1m_kline = pd.DataFrame()
            else:
                old_ohlc_1m_kline = pickle.loads(old_ohlc_1m_kline)
            new_ohlc_1m_kline = pd.concat([old_ohlc_1m_kline, ohlc_df], axis=0)
            redis_cli.set_data('ohlc_1m_kline', pickle.dumps(new_ohlc_1m_kline))
            print(f"redis ohlc_1m_kline load and saving time after appending: {time.time()-start}")
            appended_kimp_df = appended_kimp_df[appended_kimp_df['datetime_now'] >= adjusted_datetime_now]
            kline_switch = False
        elif datetime_now.second != 0:
            # INSERT into redis db for current data
            ohlc_df = generate_ohlc_df(appended_kimp_df)
            redis_cli.set_data('ohlc_1m_now', pickle.dumps(ohlc_df))
            kline_switch = True
        else:
            # INSERT into redis db for current data
            ohlc_df = generate_ohlc_df(appended_kimp_df)
            redis_cli.set_data('ohlc_1m_now', pickle.dumps(ohlc_df))

ohlc_loader_proc = Process(target=ohlc_loader, args=(redis_cli,), daemon=True)
ohlc_loader_proc.start()
    
    

generating ohlc_df(length: 18936): 0.058554649353027344
redis saving ohlc_1m_now time: 0.0005478858947753906
redis ohlc_1m_kline load and saving time after appending: 0.002550840377807617
generating ohlc_df(length: 20544): 0.048239707946777344
redis saving ohlc_1m_now time: 0.0007450580596923828
redis ohlc_1m_kline load and saving time after appending: 0.002869129180908203
generating ohlc_df(length: 20640): 0.04867291450500488
redis saving ohlc_1m_now time: 0.0004830360412597656
redis ohlc_1m_kline load and saving time after appending: 0.0026514530181884766
generating ohlc_df(length: 20784): 0.05079174041748047
redis saving ohlc_1m_now time: 0.0005357265472412109
redis ohlc_1m_kline load and saving time after appending: 0.0023953914642333984
generating ohlc_df(length: 20712): 0.04863739013671875
redis saving ohlc_1m_now time: 0.0007083415985107422
redis ohlc_1m_kline load and saving time after appending: 0.005545616149902344
generating ohlc_df(length: 20928): 0.044495344161987305
redis

In [413]:
ohlc_loader_proc.terminate()
time.sleep(1)
ohlc_loader_proc.is_alive()

False

In [411]:
pickle.loads(redis_cli.get_data('ohlc_1m_now')).reset_index()

,symbol,datetime_now,open,high,low,close
0,AAVEUSDT,2023-07-19 19:43:00,0.011174,0.011735,0.011174,0.011595
1,ALGOUSDT,2023-07-19 19:43:00,0.011977,0.011977,0.011977,0.011977
2,ANKRUSDT,2023-07-19 19:43:00,0.013050,0.013050,0.013050,0.013050
3,BCHUSDT,2023-07-19 19:43:00,0.012520,0.012520,0.012273,0.012314
4,BTCUSDT,2023-07-19 19:43:00,0.012583,0.012583,0.012144,0.012439
5,DOGEUSDT,2023-07-19 19:43:00,0.012537,0.012684,0.012390,0.012537
6,ETCUSDT,2023-07-19 19:43:00,0.012356,0.012356,0.012248,0.012302
7,ETHUSDT,2023-07-19 19:43:00,0.011711,0.011722,0.011700,0.011700
8,FLOWUSDT,2023-07-19 19:43:00,0.012907,0.014513,0.012907,0.014513
9,GMTUSDT,2023-07-19 19:43:00,0.011801,0.011801,0.010944,0.011801


In [412]:
temp = pickle.loads(redis_cli.get_data('ohlc_1m_kline')).reset_index()
temp[temp['symbol']=='BTCUSDT']

,symbol,datetime_now,open,high,low,close
4,BTCUSDT,2023-07-19 12:41:00,0.013161,0.013688,0.013161,0.013688
28,BTCUSDT,2023-07-19 12:42:00,0.013684,0.013769,0.013157,0.013160
52,BTCUSDT,2023-07-19 12:43:00,0.013160,0.013553,0.012979,0.013097
76,BTCUSDT,2023-07-19 12:44:00,0.013097,0.013235,0.012123,0.012462
100,BTCUSDT,2023-07-19 12:45:00,0.012462,0.012720,0.012378,0.012464
...,...,...,...,...,...,...
10012,BTCUSDT,2023-07-19 19:38:00,0.012108,0.012108,0.011274,0.011274
10036,BTCUSDT,2023-07-19 19:39:00,0.011274,0.011277,0.011214,0.011217
10060,BTCUSDT,2023-07-19 19:40:00,0.011217,0.011234,0.010721,0.010987
10084,BTCUSDT,2023-07-19 19:41:00,0.010987,0.011697,0.010984,0.011286


In [51]:
# Check the size of an object in Mb
import sys
sys.getsizeof(kimp_cache_df) / 1024 / 1024

32.36960315704346